# Setup

In [1]:
import os
import random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models
from torchvision.transforms import v2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, confusion_matrix
from tqdm.auto import tqdm

# Configurazione Hardware e Random Seed
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Iperparametri
IMG_SIZE = 224    # Risoluzione nativa EfficientNet
BATCH_SIZE = 32
NUM_EPOCHS = 20
LR = 1e-3

# Definizione Percorsi (Come da tue specifiche)
BASE_INPUT = "/kaggle/input/challengedlusethis"
TRAIN_IMG_DIR = os.path.join(BASE_INPUT, "final_clean_dataset/final_clean_dataset/images")
TRAIN_MASK_DIR = os.path.join(BASE_INPUT, "final_clean_dataset/final_clean_dataset/masks")
TRAIN_LABELS_CSV = os.path.join(BASE_INPUT, "final_clean_dataset/final_clean_dataset/train_labels.csv")

TEST_IMG_DIR = os.path.join(BASE_INPUT, "test_data/test_data/images")
TEST_MASK_DIR = os.path.join(BASE_INPUT, "test_data/test_data/masks")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True

set_seed(SEED)
print(f"Device: {DEVICE}")

Device: cuda


# Preparazione dati e split

In [2]:
class MaskedDataset(Dataset):
    def __init__(self, image_paths, mask_paths, labels=None, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]
        
        # Lettura Immagine
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Immagine non trovata: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Lettura Maschera e Applicazione
        if os.path.exists(mask_path):
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            # Resize di sicurezza se le dimensioni non coincidono
            if image.shape[:2] != mask.shape[:2]:
                mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
            
            # Masking: Bitwise AND mantiene i pixel dove mask > 0
            image = cv2.bitwise_and(image, image, mask=mask)
        
        # Trasformazioni
        if self.transform:
            image = self.transform(image)
            
        if self.labels is not None:
            return image, self.labels[idx]
        return image

# Trasformazioni (Augmentation Leggera + Normalizzazione ImageNet)
norm_mean = [0.485, 0.456, 0.406]
norm_std = [0.229, 0.224, 0.225]

train_transform = v2.Compose([
    v2.ToPILImage(),
    v2.Resize((IMG_SIZE, IMG_SIZE)),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    v2.Normalize(mean=norm_mean, std=norm_std)
])

val_transform = v2.Compose([
    v2.ToPILImage(),
    v2.Resize((IMG_SIZE, IMG_SIZE)),
    v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
    v2.Normalize(mean=norm_mean, std=norm_std)
])

# dataset con masking, data loaders e bilanciamento

In [3]:
print("Caricamento e verifica dataset...")

if not os.path.exists(TRAIN_LABELS_CSV):
    raise FileNotFoundError(f"CSV non trovato: {TRAIN_LABELS_CSV}")

df = pd.read_csv(TRAIN_LABELS_CSV)
valid_img_paths = []
valid_mask_paths = []
valid_labels = []

for idx, row in df.iterrows():
    fname = row['sample_index']
    # Normalizzazione nome file
    if not fname.endswith('.png'): fname += '.png'
    
    img_p = os.path.join(TRAIN_IMG_DIR, fname)
    mask_p = os.path.join(TRAIN_MASK_DIR, fname)
    
    if os.path.exists(img_p):
        valid_img_paths.append(img_p)
        valid_mask_paths.append(mask_p) # Aggiungiamo il path maschera anche se non esiste (gestito nel Dataset)
        valid_labels.append(row['label'])

print(f"File trovati: {len(valid_img_paths)} su {len(df)}")
if len(valid_img_paths) == 0:
    raise ValueError("Nessuna immagine trovata! Controlla i percorsi in TRAIN_IMG_DIR.")

# Label Encoding
le = LabelEncoder()
y_encoded = le.fit_transform(valid_labels)

# Split Stratificato
X_train, X_val, y_train, y_val, m_train, m_val = train_test_split(
    valid_img_paths, y_encoded, valid_mask_paths,
    test_size=0.2, stratify=y_encoded, random_state=SEED
)

# Setup Dataset & Loader
train_ds = MaskedDataset(X_train, m_train, y_train, transform=train_transform)
val_ds = MaskedDataset(X_val, m_val, y_val, transform=val_transform)

# Sampler per Class Imbalance
class_counts = np.bincount(y_train)
class_weights = 1. / (class_counts + 1e-6)
sample_weights = [class_weights[y] for y in y_train]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

Caricamento e verifica dataset...
File trovati: 1239 su 1239


# Modello (efficientnet B0)

In [4]:
def get_model(num_classes):
    print("Inizializzazione EfficientNet-B0...")
    model = models.efficientnet_b0(weights='DEFAULT')
    
    # Scongeliamo tutto per permettere un adattamento completo (Full Fine-Tuning)
    for param in model.parameters():
        param.requires_grad = True
        
    # Sostituzione Classifier Head
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

# training loop, plots and confusion matrix

In [5]:
model = get_model(len(le.classes_)).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, steps_per_epoch=len(train_loader), epochs=NUM_EPOCHS, pct_start=0.3)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler('cuda')

history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}
best_f1 = 0.0
best_preds, best_labels = [], []

print("Avvio Training...")
for epoch in range(NUM_EPOCHS):
    model.train()
    t_loss = 0.0
    t_preds, t_labels = [], []
    
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            out = model(imgs)
            loss = criterion(out, lbls)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        t_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(out, 1)
        t_preds.extend(preds.cpu().numpy())
        t_labels.extend(lbls.cpu().numpy())
        
    t_loss /= len(train_loader.dataset)
    t_f1 = f1_score(t_labels, t_preds, average='weighted')
    
    # Validation
    model.eval()
    v_loss = 0.0
    v_preds, v_labels = [], []
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            out = model(imgs)
            loss = criterion(out, lbls)
            
            v_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(out, 1)
            v_preds.extend(preds.cpu().numpy())
            v_labels.extend(lbls.cpu().numpy())
            
    v_loss /= len(val_loader.dataset)
    v_f1 = f1_score(v_labels, v_preds, average='weighted')
    
    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_f1'].append(t_f1)
    history['val_f1'].append(v_f1)
    
    print(f"Ep {epoch+1:02d} | T-Loss: {t_loss:.4f} F1: {t_f1:.4f} | V-Loss: {v_loss:.4f} F1: {v_f1:.4f}")
    
    if v_f1 > best_f1:
        best_f1 = v_f1
        best_preds = v_preds
        best_labels = v_labels
        torch.save(model.state_dict(), "best_model.pth")
        print("--> Best Model Saved")

print(f"Training concluso. Best Val F1: {best_f1:.4f}")
plot_results(history, best_labels, best_preds, le.classes_)

Inizializzazione EfficientNet-B0...


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 120MB/s] 


Avvio Training...
Ep 01 | T-Loss: 1.3989 F1: 0.2562 | V-Loss: 1.4066 F1: 0.2176
--> Best Model Saved
Ep 02 | T-Loss: 1.3718 F1: 0.3071 | V-Loss: 1.3974 F1: 0.2357
--> Best Model Saved
Ep 03 | T-Loss: 1.3115 F1: 0.3822 | V-Loss: 1.4839 F1: 0.2307
Ep 04 | T-Loss: 1.2569 F1: 0.4360 | V-Loss: 1.4532 F1: 0.3065
--> Best Model Saved
Ep 05 | T-Loss: 1.2244 F1: 0.4981 | V-Loss: 1.5483 F1: 0.2563
Ep 06 | T-Loss: 1.2322 F1: 0.4959 | V-Loss: 1.5705 F1: 0.1853
Ep 07 | T-Loss: 1.1898 F1: 0.5088 | V-Loss: 1.6855 F1: 0.2427
Ep 08 | T-Loss: 1.1229 F1: 0.5464 | V-Loss: 1.6370 F1: 0.2810
Ep 09 | T-Loss: 1.0809 F1: 0.5920 | V-Loss: 1.7111 F1: 0.2979
Ep 10 | T-Loss: 1.0246 F1: 0.6236 | V-Loss: 1.5806 F1: 0.3023
Ep 11 | T-Loss: 0.9495 F1: 0.6501 | V-Loss: 1.8283 F1: 0.3283
--> Best Model Saved
Ep 12 | T-Loss: 0.8900 F1: 0.7152 | V-Loss: 1.6842 F1: 0.3259
Ep 13 | T-Loss: 0.8121 F1: 0.7661 | V-Loss: 1.8046 F1: 0.3399
--> Best Model Saved
Ep 14 | T-Loss: 0.7107 F1: 0.8237 | V-Loss: 1.8952 F1: 0.3308
Ep 15 | T

NameError: name 'plot_results' is not defined

# generazione submission 

In [ ]:
def create_submission(model_path, output_csv="submission.csv"):
    print("\nGenerazione Submission...")
    model.load_state_dict(torch.load(model_path))
    model.eval()
    
    test_files = [f for f in os.listdir(TEST_IMG_DIR) if f.endswith('.png')]
    # Ordinamento numerico importante per coerenza
    test_files.sort(key=lambda x: int(x.split('_')[1].split('.')[0]))
    
    t_base = v2.Compose([
        v2.ToPILImage(), v2.Resize((IMG_SIZE, IMG_SIZE)),
        v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]),
        v2.Normalize(mean=norm_mean, std=norm_std)
    ])
    
    predictions = []
    
    with torch.no_grad():
        for i, fname in enumerate(test_files):
            img_path = os.path.join(TEST_IMG_DIR, fname)
            mask_path = os.path.join(TEST_MASK_DIR, fname)
            
            # Caricamento
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Masking (se la maschera esiste nel test set)
            if os.path.exists(mask_path):
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                if image.shape[:2] != mask.shape[:2]:
                    mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
                image = cv2.bitwise_and(image, image, mask=mask)
            
            # TTA (Base + Flip)
            input_tensor = t_base(image)
            input_fh = t_base(cv2.flip(image, 1))
            
            batch = torch.stack([input_tensor, input_fh]).to(DEVICE)
            
            logits = model(batch)
            probs = torch.softmax(logits, dim=1)
            avg_prob = torch.mean(probs, dim=0)
            
            pred_idx = torch.argmax(avg_prob).item()
            pred_label = le.inverse_transform([pred_idx])[0]
            
            # sample_index deve includere l'estensione nel csv di sottomissione?
            # Solitamente nella challenge an2dl è richiesto 'img_XXXX.png'.
            # Usiamo fname così com'è.
            predictions.append((fname, pred_label))
            
            if (i+1) % 200 == 0: print(f"Processati {i+1}...")

    df_sub = pd.DataFrame(predictions, columns=['sample_index', 'label'])
    df_sub.to_csv(output_csv, index=False)
    print(f"✅ Submission salvata in {output_csv}")

create_submission("best_model.pth")